# 🛡️ Sudarshana (StegShield) — Account 1 Training Pipeline
### High-Speed T4 GPU Training on Real-World Cyber Threats & Steganography

This notebook trains the Sudarshana Steganalyzer on **NVIDIA T4 GPU**.
Follow the cells in order: **1 → 2 → 3 → 4 → 5 → 6**.

In [ ]:
# ── Cell 1: Check GPU Acceleration ──
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# ── Cell 2: Extract Codebase & Install Requirements ──
import os, sys

# If EDI_Project_Colab_Latest.zip was uploaded to /content or Google Drive:
if os.path.exists('/content/EDI_Project_Colab_Latest.zip'):
    ZIP_PATH = '/content/EDI_Project_Colab_Latest.zip'
elif os.path.exists('/content/drive/MyDrive/EDI_Project_Colab_Latest.zip'):
    ZIP_PATH = '/content/drive/MyDrive/EDI_Project_Colab_Latest.zip'
else:
    from google.colab import files
    print("Please select and upload EDI_Project_Colab_Latest.zip:")
    uploaded = files.upload()
    ZIP_PATH = list(uploaded.keys())[0]

!unzip -q -o "{ZIP_PATH}" -d /content/EDI_Project
%cd /content/EDI_Project
sys.path.insert(0, 'src')
!pip install -q scipy scikit-learn seaborn plotly
print("✅ Workspace ready at /content/EDI_Project")

In [ ]:
# ── Cell 3: Setup Dataset & Ingest Cyber Threats ──
import os
from pathlib import Path

DRIVE_CLEAN = "/content/drive/MyDrive/EDI_Project_Final/data/clean"
DRIVE_STEGO = "/content/drive/MyDrive/EDI_Project_Final/data/stego"

!mkdir -p data/clean data/stego

if os.path.exists(DRIVE_CLEAN) and os.path.exists(DRIVE_STEGO):
    print("⚡ Copying 10,000 image dataset from Google Drive to local SSD...")
    !cp -r "{DRIVE_CLEAN}"/* data/clean/
    !cp -r "{DRIVE_STEGO}"/* data/stego/
else:
    print("ℹ️ Drive dataset not detected. Generating procedural cover images & synthetic stego...")
    !python scripts/generate_malicious_stego.py --num_images 150 --add_to_stego

# Copy 2024-2025 modern cyberattack carriers into training set
!cp -f data/malicious_eval/*.png data/stego/ 2>/dev/null || true

clean_count = len(list(Path("data/clean").glob("*.*")))
stego_count = len(list(Path("data/stego").glob("*.*")))
print(f"\n📊 Dataset Ready:")
print(f"   Clean Covers:   {clean_count} images")
print(f"   Stego Carriers: {stego_count} images")

In [ ]:
# ── Cell 4: Start GPU Training (Account 1) ──
# Uses Joint Fusion mode with AdamW + Cosine Annealing scheduler & fp16 mixed precision
!python src/training/train.py \
    --mode joint \
    --epochs 30 \
    --lr 1e-4 \
    --batch_size 32 \
    --seed 42

In [ ]:
# ── Cell 5: Evaluate on Real-World Cyber Threats & Test Accuracy ──
print("🔍 Evaluating against real-world 2024-2025 malware carriers...")
!python scripts/generate_malicious_stego.py --evaluate

# Calculate final test set accuracy
!python -c "
import torch
from src.model.fusion_model import MultiBranchSteganalyzer
from src.training.config import Config
from src.data_pipeline.dataset import get_dataloaders
from src.utils.checkpoints import extract_state_dict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = Config()
cfg.batch_size = 32
_, _, test_loader = get_dataloaders(cfg)

model = MultiBranchSteganalyzer().to(device)
raw = torch.load('weights/fusion_best.pt', map_location=device)
model.load_state_dict(extract_state_dict(raw), strict=False)
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

acc = (correct / total) * 100
print(f'\\n🏆 Final Model Test Accuracy: {acc:.2f}% ({correct}/{total})')
"

In [ ]:
# ── Cell 6: Download & Save Trained Weights ──
import os, shutil
from google.colab import files

# 1. Save to Google Drive if connected:
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy2('weights/fusion_best.pt', '/content/drive/MyDrive/fusion_best_account1.pt')
    print("✅ Saved backup to Google Drive: fusion_best_account1.pt")

# 2. Direct browser download to your Mac:
print("📥 Triggering browser download of trained weights...")
files.download('weights/fusion_best.pt')